In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [19]:
df = pd.read_csv("car_cleaned_ready.csv")

In [20]:
df.head()

,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb,CarAge,Km_per_year,Is_Urban,LogPrice
0,1500.0,137830.0,4.0,1,1998,0,0,28,4922.500000,1,7.313887
1,4171.0,128548.0,4.0,0,2016,1,0,10,12854.800000,0,8.336151
2,5331.0,107302.0,4.0,0,2014,0,1,12,8941.833333,1,8.581482
3,1500.0,141838.0,4.0,1,1999,0,1,27,5253.259259,1,7.313887
4,1500.0,128548.0,3.0,0,2022,0,0,4,32137.000000,1,7.313887


In [21]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    int64  
 4   Year             140 non-null    int64  
 5   Location_Rural   140 non-null    int64  
 6   Location_Suburb  140 non-null    int64  
 7   CarAge           140 non-null    int64  
 8   Km_per_year      140 non-null    float64
 9   Is_Urban         140 non-null    int64  
 10  LogPrice         140 non-null    float64
dtypes: float64(5), int64(6)
memory usage: 12.2 KB


,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb,CarAge,Km_per_year,Is_Urban,LogPrice
count,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000
mean,3175.456250,130945.403571,3.785714,0.721429,2010.235714,0.150000,0.364286,15.764286,12040.312842,0.850000,7.797661
std,2601.848629,53815.006935,0.846369,0.882017,7.280219,0.358354,0.482957,7.280219,11603.322917,0.358354,0.684154
min,1500.000000,5000.000000,2.000000,0.000000,1998.000000,0.000000,0.000000,3.000000,294.117647,0.000000,7.313887
25%,1500.000000,97844.000000,3.000000,0.000000,2004.000000,0.000000,0.000000,9.750000,5190.796131,1.000000,7.313887
50%,1500.000000,128548.000000,4.000000,0.000000,2010.000000,0.000000,0.000000,16.000000,8818.769608,1.000000,7.313887
75%,4489.750000,167501.500000,4.000000,1.000000,2016.250000,0.000000,1.000000,22.000000,14415.663462,1.000000,8.409743
max,8974.375000,271987.750000,5.000000,3.000000,2023.000000,1.000000,1.000000,28.000000,74446.000000,1.000000,9.102240


In [22]:
#STEP-1 Choose target:

y = df["LogPrice"]
X = df.drop(columns=["Price", "LogPrice"])

In [23]:
#STEP-2  Encoding

X = pd.get_dummies(X, drop_first=True)

In [24]:
df.head()

,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb,CarAge,Km_per_year,Is_Urban,LogPrice
0,1500.0,137830.0,4.0,1,1998,0,0,28,4922.500000,1,7.313887
1,4171.0,128548.0,4.0,0,2016,1,0,10,12854.800000,0,8.336151
2,5331.0,107302.0,4.0,0,2014,0,1,12,8941.833333,1,8.581482
3,1500.0,141838.0,4.0,1,1999,0,1,27,5253.259259,1,7.313887
4,1500.0,128548.0,3.0,0,2022,0,0,4,32137.000000,1,7.313887


In [25]:
X = pd.get_dummies(X, drop_first=True)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [27]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [28]:
lr = LinearRegression()

lr_cv_r2 = cross_val_score(lr, X_train, y_train, cv=kf, scoring="r2")
lr_cv_rmse = cross_val_score(lr, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")

print("Linear Regression CV Results")
print("R2 Mean:", lr_cv_r2.mean())
print("RMSE Mean:", -lr_cv_rmse.mean())

Linear Regression CV Results
R2 Mean: 0.5371567500363649
RMSE Mean: 0.4474289074039504


In [29]:
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [30]:
ridge = Ridge(alpha=1.0)

ridge_cv_r2 = cross_val_score(ridge, X_train, y_train, cv=kf, scoring="r2")
ridge_cv_rmse = cross_val_score(ridge, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")

print("Ridge Regression CV Results")
print("R2 Mean:", ridge_cv_r2.mean())
print("RMSE Mean:", -ridge_cv_rmse.mean())

Ridge Regression CV Results
R2 Mean: 0.5392820674287652
RMSE Mean: 0.4464555888848391


In [31]:
ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)

In [32]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_cv_r2 = cross_val_score(rf, X_train, y_train, cv=kf, scoring="r2")
rf_cv_rmse = cross_val_score(rf, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")

print("Random Forest CV Results")
print("R2 Mean:", rf_cv_r2.mean())
print("RMSE Mean:", -rf_cv_rmse.mean())

Random Forest CV Results
R2 Mean: 0.7072453667706556
RMSE Mean: 0.35694025655392253


In [33]:
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [36]:
def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    print(f"\n{name}")
    print("R2  :", round(r2, 3))
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))

In [37]:
evaluate("Linear Regression", y_test, lr_pred)
evaluate("Ridge Regression", y_test, ridge_pred)
evaluate("Random Forest", y_test, rf_pred)


Linear Regression
R2  : 0.541
MAE : 0.361
RMSE: 0.466

Ridge Regression
R2  : 0.542
MAE : 0.36
RMSE: 0.466

Random Forest
R2  : 0.479
MAE : 0.257
RMSE: 0.497


In [38]:
i = 5

x_one = X_test.iloc[[i]]
y_true = y_test.iloc[i]

print("Actual:", y_true)

print("LR:", lr.predict(x_one)[0])
print("Ridge:", ridge.predict(x_one)[0])
print("RF:", rf.predict(x_one)[0])

Actual: 8.195057690895077
LR: 8.169710994020214
Ridge: 8.168208409919501
RF: 8.016684254614724
